In [ ]:
# !pip install emoji

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Thesis_Repository/Final_Google_Drive') # change directory to the current working directory

In [ ]:
import pandas as pd
import json
import numpy as np
from IPython.display import display
from pathlib import Path
import os


STEPVERIFY_PATH = Path("./original_data/stepverify.json")
MATHDIAL_OUT_PATH = Path("./original_data/")

if not MATHDIAL_OUT_PATH.exists():
  MATHDIAL_OUT_PATH.mkdir(parents=True, exist_ok=True)



In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import login
notebook_login()
login(token="[HF Token]")

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.jsonl', 'test': 'test.jsonl'}
mathdial_train_df = pd.read_json("hf://datasets/eth-nlped/mathdial/" + splits["train"], lines=True)
mathdial_test_df = pd.read_json("hf://datasets/eth-nlped/mathdial/" + splits["test"], lines=True)

# concat train and test sets
mathdial_df = pd.concat([mathdial_train_df, mathdial_test_df], ignore_index=True)
mathdial_df.to_csv(MATHDIAL_OUT_PATH / "mathdial_df.csv", index=False)

In [ ]:
with open(STEPVERIFY_PATH, "r", encoding="utf-8") as f:
    stepverify_data = json.load(f)

stepverify_df = pd.DataFrame(stepverify_data)

In [ ]:
# stepverify_df = stepverify_df[["problem", "student_incorrect_solution", "dialog_history"]].copy()
# stepverify_df.rename(columns={"dialog_history": "conversation", "problem": "question"}, inplace=True)
# stepverify_df['conversation'][0]

### Prepare Mathdial Dataframe with contexts to compare with Stepverify Dataset
- normalize questions
- parse conversations into turns ( same format as Stepverify )
- student_turns : to be used to calculate the scores between two conversations from MathDial and Stepverify at a conversation level
- teacher_turn_context : teacher turn utterance with preceding 1 and following 1 student utterances


In [ ]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import normalize_text
from typing import Any
import re

In [ ]:
mathdial_df["conversation"][0]

In [ ]:
def parse_mathdial_conversation(conversation: Any) -> list[dict[str, Any]]:
    turns: list[dict[str, Any]] = []

    for raw_turn in conversation.split("|EOM|"):
        turn = raw_turn.strip()
        if not turn or ":" not in turn:
            continue

        speaker, text = turn.split(":", 1) # split the turn into speaker and text ( max 2 items)
        speaker_key = speaker.strip().lower()
        text = text.strip()
        label = None

        if speaker_key == "teacher":
            label_match = re.match(r"^\s*\(([^()]+)\)\s*(.*)$", text, flags=re.DOTALL) # (..)text, include dots

            if label_match:
                label = label_match.group(1).strip().lower()
                text = label_match.group(2).strip()
            else:
                label = None
                text = text

            user = "teacher"

        else:
            user = "student"
            label = None
            text = text

        turns.append(
            {
                "turn_idx": len(turns),
                "user": normalize_text(user),
                "label": normalize_text(label),
                # "text": text,
                "text": normalize_text(text),
            }
        )

    return turns


In [ ]:
# md_df["conversation"][0]

In [ ]:
# md_df["parsed_conversation"] = md_df["conversation"].apply(parse_mathdial_conversation)
# md_df["parsed_conversation"][0]

In [ ]:
def find_mathdial_neighbor_turn(
    turns: list[dict[str, Any]],
    turn_idx: int,
    direction: int,
    target_user:str
) -> dict[str, Any] | None:
    """ Fine the closest preceding/following turn with the requested user
    Args:
        turns (list): conversation turns. each turn has a user, text, and label if the user is teacher
        turn_idx (int): starting index in the conversation turns
        direction (int): 1 for following turn, -1 for preceding turn
        target_user (str): "teacher" or "student"

    Returns:
        the closest turn with the requested user and the turn text. if the turn is at first or last index, return None
    """
    neighbor_idx = turn_idx + direction

    while 0 <= neighbor_idx < len(turns):
        candidate_turn = turns[neighbor_idx]
        if candidate_turn["user"] == target_user:
            return candidate_turn
        neighbor_idx += direction
    return None



def construct_mathdial_teacher_turn_context(turns: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    Construct the teacher turn context with preceding and following student turns.

    Args:
        turns (list[dict[str, Any]]): List of conversation turns. Each turn is a dictionary
            containing at least the keys "user", "text", "label" (label is present if the user is teacher), and "turn_idx".

    Returns:
        list[dict[str, Any]]: A list of dictionaries. Each dictionary represents a teacher turn
            and contains the teacher's turn index ("turn_idx"), label ("label"), text ("text"),
            previous student's text ("prev_student"), and next student's text ("next_student").
            If a previous or next student turn does not exist, its value is None.
    """
    teacher_turn_contexts: list[dict[str, Any]] = []


    for turn_idx, turn in enumerate(turns):
        if turn["user"] != "teacher":
            continue

        prev_student = find_mathdial_neighbor_turn(turns, turn_idx, direction=-1, target_user="student")
        next_student = find_mathdial_neighbor_turn(turns, turn_idx, direction=1, target_user="student")

        teacher_turn_contexts.append(
            {
                "turn_idx": int(turn["turn_idx"]),
                "label": normalize_text(turn["label"]),
                "text": normalize_text(turn["text"]),
                "prev_student": None if prev_student is None else normalize_text(prev_student["text"]),
                "next_student": None if next_student is None else normalize_text(next_student["text"]),
            }
        )

    return teacher_turn_contexts


def construct_mathdial_dataframe_with_context(md_df: pd.DataFrame) -> pd.DataFrame:
    """ Add teacher turn context column, student turns column, and a column for the number of turns (the whole conversation, students turns, and teacher turns )
        These new columns will be used to calculate the scores between MathDial and Stepverify to determine the teacher labels in Stepverify Dataset
        Args:
            md_df (pd.DataFrame): MathDial DataFrame with parsed conversation
        Returns:
            pd.DataFrame: MathDial DataFrame with teacher turn context, student turns, and the number of turns
    """
    md_df_with_context = md_df.reset_index(drop=True).copy()

    md_df_with_context.insert(0, "mathdial_row_id", np.arange(len(md_df_with_context))) # add mathdial_row_id column in the first column
    md_df_with_context["parsed_conversation_turns"] = md_df_with_context["conversation"].apply(parse_mathdial_conversation)
    md_df_with_context["student_turns"] = md_df_with_context["parsed_conversation_turns"].apply(lambda turns: [normalize_text(turn["text"]) for turn in turns if turn["user"] == "student"])
    md_df_with_context["teacher_turn_contexts"] = md_df_with_context["parsed_conversation_turns"].apply(construct_mathdial_teacher_turn_context)
    md_df_with_context["question"] = md_df_with_context["question"].apply(normalize_text)
    md_df_with_context["student_incorrect_solution"] = md_df_with_context["student_incorrect_solution"].apply(normalize_text)

    md_df_with_context["n_parsed_conversation_turns"] = md_df_with_context["parsed_conversation_turns"].apply(len)
    md_df_with_context["n_student_turns"] = md_df_with_context["student_turns"].apply(len)
    md_df_with_context["n_teacher_turn_contexts"] = md_df_with_context["teacher_turn_contexts"].apply(len)

    return md_df_with_context


def load_and_contstruc_mathdial_dataframe_with_context(md_df: pd.DataFrame) -> pd.DataFrame:
    columns_to_use = ["question", "student_incorrect_solution", "conversation"]
    md_df = md_df[columns_to_use].copy()
    md_df_with_context = construct_mathdial_dataframe_with_context(md_df)

    final_columns_to_use = ["mathdial_row_id", "question", "student_incorrect_solution", "parsed_conversation_turns", "student_turns", "teacher_turn_contexts", "n_parsed_conversation_turns", "n_student_turns", "n_teacher_turn_contexts"]
    md_df_with_context = md_df_with_context[final_columns_to_use]

    return md_df_with_context

In [ ]:
md_df_with_context = load_and_contstruc_mathdial_dataframe_with_context(mathdial_df)
md_df_with_context.head()

In [ ]:
md_df_with_context["teacher_turn_contexts"][0]

In [ ]:
# md_df_with_context["student_turns"][0]

## Build the Mathdial Question Lookup Table

In [ ]:
from collections import defaultdict

def build_mathdial_question_lookup_table(md_df_with_context: pd.DataFrame) -> tuple[dict[str, list[int]], dict[int, dict[str, Any]]]:
    """
        Mathdial dataset has multiple rows with the same question.
        This function builds a lookup table that maps each question to a list of row IDs.

        Args:
            md_df_with_context (pd.DataFrame): MathDial DataFrame with parsed conversation
        Returns:
            tuple[dict[str, list[int]], dict[int, dict[str, Any]]]: A tuple of two dictionaries.
                The first dictionary maps each question to a list of row IDs.
                The second dictionary maps each row ID to a dictionary of row data and corresponding question.
    """
    question_index: dict[str, list[int]] = defaultdict(list)
    row_lookup: dict[int, dict[str, Any]] = {}

    for record in md_df_with_context.to_dict(orient="records"):
        row_id = int(record["mathdial_row_id"])
        question = record["question"]

        question_index[question].append(row_id)
        row_lookup[row_id] = record

    return dict(question_index), row_lookup

mathdial_question_index, mathdial_row_lookup = build_mathdial_question_lookup_table(md_df_with_context)

In [ ]:
print(mathdial_question_index)

In [ ]:
mathdial_row_lookup[0]

## Prepare Stepverify Dataframe with context

In [ ]:
stepverify_df.head()

In [ ]:
def find_stepverify_neighbor_turn(
    turns: list[dict[str, Any]],
    turn_idx: int,
    direction: int,
    target_user:str
) -> dict[str, Any] | None:
    """ Fine the closest preceding/following turn with the requested user
    Args:
        turns (list): conversation turns. each turn has a user, text, and label if the user is teacher
        turn_idx (int): starting index in the conversation turns
        direction (int): 1 for following turn, -1 for preceding turn
        target_user (str): "teacher" or "student"

    Returns:
        the closest turn with the requested user and the turn text. if the turn is at first or last index, return None
    """
    neighbor_idx = turn_idx + direction

    while 0 <= neighbor_idx < len(turns):
        candidate_turn = turns[neighbor_idx]
        if candidate_turn["user"] == target_user:
            return candidate_turn
        neighbor_idx += direction
    return None


def construct_stepverify_teacher_turn_context(turns: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    Construct the teacher turn context with preceding and following student turns.

    Args:
        turns (list[dict[str, Any]]): List of conversation turns. Each turn is a dictionary
            containing at least the keys "user", "text", "label" (label is present if the user is teacher), and "turn_idx".

    Returns:
        list[dict[str, Any]]: A list of dictionaries. Each dictionary represents a teacher turn
            and contains the teacher's turn index ("turn_idx"), text ("text"),
            previous student's text ("prev_student"), and next student's text ("next_student").
            If a previous or next student turn does not exist, its value is None.
    """
    teacher_turn_contexts: list[dict[str, Any]] = []


    for turn_idx, turn in enumerate(turns):
        if turn["user"] != "teacher":
            continue

        prev_student = find_stepverify_neighbor_turn(turns, turn_idx, direction=-1, target_user="student")
        next_student = find_stepverify_neighbor_turn(turns, turn_idx, direction=1, target_user="student")

        teacher_turn_contexts.append(
            {
                "turn_idx": turn["turn_idx"],  # teacher turn index
                "text": normalize_text(turn["text"]),  # teacher turn text
                "prev_student": None if prev_student is None else normalize_text(prev_student["text"]),
                "next_student": None if next_student is None else normalize_text(next_student["text"]),
            }
        )
    return teacher_turn_contexts


def construct_mathdial_dataframe_with_context(sv_df: pd.DataFrame) -> pd.DataFrame:
    """ Add teacher turn context column, student turns column, and a column for the number of turns (the whole conversation, students turns, and teacher turns )
        These new columns will be used to calculate the scores between MathDial and Stepverify to determine the teacher labels in Stepverify Dataset
        Args:
            md_df (pd.DataFrame): MathDial DataFrame with parsed conversation
        Returns:
            pd.DataFrame: MathDial DataFrame with teacher turn context, student turns, and the number of turns
    """
    sv_df_with_context = sv_df.reset_index(drop=True).copy()

    sv_df_with_context.insert(0, "stepverify_row_id", np.arange(len(sv_df_with_context))) # add mathdial_row_id column in the first column

    sv_df_with_context["conversation"] = sv_df_with_context["conversation"].apply(lambda turns: turns[-3:]) # Keep only the last three turns of conversations and reset index, sanity check. VERY IMPORTANT

    sv_df_with_context["parsed_conversation_turns"] = sv_df_with_context["conversation"].apply(lambda lst: [{**turn, "turn_idx": idx} for idx, turn in enumerate(lst)]) # add turn_idx to each turn to be in the same format as MathDial

    # After parsing the turns, the distribution of the number of turns is as follows -> 3: 967, 4: 29, 5: 4, 6: 2
    # Inspection of the turns showed that the last three turns are the valid utterances. Thus, here, the last three turns are used.
    sv_df_with_context["parsed_conversation_turns"] = (sv_df_with_context["conversation"].apply(lambda turns: [{**turn, "turn_idx": idx} for idx, turn in enumerate(turns[-3:])])) # Keep only the last three turns of conversations and reset index

    sv_df_with_context["parsed_conversation_turns"] = sv_df_with_context["parsed_conversation_turns"].apply(lambda turns: [{normalize_text(k) if k != "turn_idx" else k: normalize_text(v) for k, v in turn.items()} for turn in turns]) # normalize the text and keys

    sv_df_with_context["student_turns"] = sv_df_with_context["parsed_conversation_turns"].apply(lambda turns: [normalize_text(turn["text"]) for turn in turns if turn["user"] == "student"])
    sv_df_with_context["teacher_turn_contexts"] = sv_df_with_context["parsed_conversation_turns"].apply(construct_stepverify_teacher_turn_context)
    sv_df_with_context["question"] = sv_df_with_context["question"].apply(normalize_text)
    sv_df_with_context["student_incorrect_solution"] = sv_df_with_context["student_incorrect_solution"].apply(str).apply(normalize_text) # student_incorrect_solution is a list of strings, so we need to convert it to a string first

    sv_df_with_context["n_parsed_conversation_turns"] = sv_df_with_context["parsed_conversation_turns"].apply(len)
    sv_df_with_context["n_student_turns"] = sv_df_with_context["student_turns"].apply(len)
    sv_df_with_context["n_teacher_turn_contexts"] = sv_df_with_context["teacher_turn_contexts"].apply(len)


    return sv_df_with_context


def load_and_contstruc_stepverify_dataframe_with_context(stepverify_df: pd.DataFrame) -> pd.DataFrame:
    # sv_df = stepverify_df[["problem", "student_incorrect_solution", "dialog_history"]].copy()
    sv_df = stepverify_df.copy()
    sv_df.rename(columns={"dialog_history": "conversation", "problem": "question"}, inplace=True)
    sv_df_with_context = construct_mathdial_dataframe_with_context(sv_df)

    # final_columns_to_use = ["stepverify_row_id", "question", "student_incorrect_solution", "parsed_conversation_turns", "student_turns", "teacher_turn_contexts", "n_parsed_conversation_turns", "n_student_turns", "n_teacher_turn_contexts"]
    # sv_df_with_context = sv_df_with_context[final_columns_to_use]

    return sv_df_with_context

sv_df_with_context = load_and_contstruc_stepverify_dataframe_with_context(stepverify_df)


In [ ]:
sv_df_with_context.head(2)

In [ ]:
sv_df_with_context['conversation'][0]

In [ ]:
sv_df_with_context["student_turns"][0]

## Stepverify Conversation Sanity Check
- conversations with more than 3 turns can be seen as anomaly
    - in case a conversation has more than three turns, the last three turns in conversations are the valid utterances.
  -  in ```construct_mathdial_dataframe_with_context``` function -> ```sv_df_with_context["conversation"] = sv_df_with_context["conversation"].apply(lambda turns: turns[-3:])``` added to avoid abnormal conversation data

In [ ]:
sv_df_with_context["n_parsed_conversation_turns"].value_counts().sort_index()

##### Conversations with more than 3 turns

In [ ]:
import pandas as pd
from pprint import pprint

pd.reset_option("display.max_colwidth")

more_than_three_turns_mask = sv_df_with_context['n_parsed_conversation_turns'] > 3
# To sort by 'n_parsed_conversation_turns', select it as well and sort, then get the column
more_than_three_turns_stepverify_df = sv_df_with_context.loc[more_than_three_turns_mask, ['parsed_conversation_turns', 'n_parsed_conversation_turns']]
more_than_three_turns_stepverify_df = more_than_three_turns_stepverify_df.sort_values("n_parsed_conversation_turns")



for idx, row in more_than_three_turns_stepverify_df.iterrows():
    pprint(row["n_parsed_conversation_turns"])
    pprint(row["parsed_conversation_turns"])
    print("--------------")

print(len(more_than_three_turns_stepverify_df))
print(more_than_three_turns_mask.sum())

In [ ]:
### Pattern check
more_than_three_turns_pattern = more_than_three_turns_stepverify_df["parsed_conversation_turns"].apply(
    lambda turns: tuple(turn["user"] for turn in turns)
)

print(more_than_three_turns_pattern.unique())

In [ ]:
### last three turns
more_than_three_turns__last_three_patterns = more_than_three_turns_stepverify_df["parsed_conversation_turns"].apply(
    lambda turns: tuple(turn["user"] for turn in turns[-3:])
)

print(more_than_three_turns__last_three_patterns.unique())

##### Conversations with 3 turns

In [ ]:
import pandas as pd

pd.reset_option("display.max_colwidth")

three_turns_mask = sv_df_with_context['n_parsed_conversation_turns'] == 3
# To sort by 'n_parsed_conversation_turns', select it as well and sort, then get the column
three_turns_stepverify_df = sv_df_with_context.loc[three_turns_mask, ['parsed_conversation_turns', 'n_parsed_conversation_turns']]
three_turns_stepverify_df = three_turns_stepverify_df.sort_values("n_parsed_conversation_turns")

# for idx, row in three_turns_stepverify_df.iterrows():
#     pprint(row["n_parsed_conversation_turns"])
#     pprint(row["parsed_conversation_turns"])
#     print("--------------")

print(len(three_turns_stepverify_df))

In [ ]:
three_turns__pattern = three_turns_stepverify_df["parsed_conversation_turns"].apply(lambda turns: tuple(turn["user"] for turn in turns))

print(three_turns__pattern.unique())

## 6. Scoring Fnctions


1. ```score_student_turn_similarity```

- Among MathDial candidates with the same problem, select the conversation in which the student utterances are the most similar.
    - If both data points have student utterances: Calculate the average similarity of student utterances in the same order.
    - If one does not have student utterances: Use the similarity of the `student_incorrect_solution` field. --> just a fallback, not used in the final result

2. ```score_teacher_turn_candidate```
- Each dataset contains teacher_turn_context column, which consists of teacher turns along with preceding and following 1 turn student utterances.
- This function computes the mean similarity of all three utterances.


The similarity calculation is done by the function ```SequenceMatcher```, which compares two sentences characterwise.
- StepVerify dataset itself is a subset of mathdial dataset, which does not paraphrase the original text much.
- So comparing the similiarities at the surface level, instead of sementic level, worked very well for this task.

In [ ]:
# print(sv_df_with_context["student_turns"][0])
# print(md_df_with_context["student_turns"][0] )

In [ ]:
import string
from difflib import SequenceMatcher

def calculate_similarity_between_two_strings(a: str, b: str) -> float:
    if not a or not b:
        return 0.0

    elif a == b:
        return 1.0
    else:
        score = SequenceMatcher(None, a, b).ratio() # Stepverify dataset can be seen as a subset of MathDial dataset. Embedding models capture semantics better, but character-level matching can be more precise for near-exact step comparison.

    return score


def score_student_turn_similarity(
    sv_record: dict[str, Any],
    md_record: dict[str, Any],
) -> tuple[float, str, list[float]]:
    """
        Score a StepVerify dialogue against one already-parsed MathDial row.
        The similarity score is calculated using SequenceMatcher in calculate_similarity_between_two_strings function.
        The similarity score is based on character
    """

    sv_student_turns = sv_record["student_turns"]
    md_student_turns = md_record["student_turns"]

    if sv_student_turns and md_student_turns:
        # Calculate the average similarity of student utterances in the same order.
        pair_scores = [calculate_similarity_between_two_strings(sv_student, md_student_turns[min(turn_idx, len(md_student_turns) - 1)]) for turn_idx, sv_student in enumerate(sv_student_turns)] # sv_student utterance 1 & md_student utterance 1, sv_student utterance 2 & md_student utterance 2, ...
        return float(np.mean(pair_scores)), "student_turns", pair_scores

    # # if there is no student utterances in both datasets, use the similarity of the `student_incorrect_solution` field.
    # # used to avoid any errors
    # fallback_score = calculate_similarity_between_two_strings(sv_record["student_incorrect_solution"], md_record["student_incorrect_solution"])
    # return fallback_score, "incorrect_solution", [fallback_score]


def score_teacher_turn_candidate(
    sv_teacher_context: dict[str, Any],
    md_teacher_context: dict[str, Any],
) -> dict[str, float | None]:
    """Score a StepVerify teacher turn against one preparsed MathDial teacher turn."""

    teacher_text_score = calculate_similarity_between_two_strings(sv_teacher_context["text"], md_teacher_context["text"]) # calculate the similarity between the teacher's text in Stepverify and MathDial

    teacher_context_score = [teacher_text_score]
    prev_student_score: float | None = None  # one student's text in Stepverify and MathDial before the teacher's text
    next_student_score: float | None = None  # one student's text in Stepverify and MathDial after the teacher's text

    # calculate the similarity between the previous student's text in Stepverify and MathDial
    if sv_teacher_context["prev_student"] and md_teacher_context["prev_student"]:
        prev_student_score = calculate_similarity_between_two_strings(sv_teacher_context["prev_student"], md_teacher_context["prev_student"])
        teacher_context_score.append(prev_student_score)

    if sv_teacher_context["next_student"] and md_teacher_context["next_student"]:
        next_student_score = calculate_similarity_between_two_strings(sv_teacher_context["next_student"], md_teacher_context["next_student"])
        teacher_context_score.append(next_student_score)

    return {
        "teacher_text_score": teacher_text_score,
        "prev_student_score": prev_student_score,
        "next_student_score": next_student_score,
        "mean_teacher_context_score": float(np.mean(teacher_context_score)), # calculate the mean of the teacher's text, previous student's text, and next student's text
    }

## 7. Stage 1 : Select the best MathDial Conversation

In [ ]:
def match_student_conversation_between_stepverify_mathdial(
    sv_df_with_context: pd.DataFrame,
    mathdial_problem_index: dict[str, list[int]],
    mathdial_row_lookup: dict[int, dict[str, Any]],
) -> pd.DataFrame:
    """ This function first looks up the mathdial question based on the questions in Stepverify.
        If the same questions are found in mathdial dataset, the student utterance turns will be compared between the current stepverify dataset row and the mathdial dataset rows with the same question.
        The result of the comparisons will be stored in "match_records" along with the current stepverify row index and mathdial row indices.
        These indices will be used to choose the best matching teacher utterances between two conversations in Mathdial and Stepverify.
    """
    match_records: list[dict[str, Any]] = []

    for sv_record in sv_df_with_context.to_dict(orient="records"): # for each row of stepverify
        stepverify_row_id = int(sv_record["stepverify_row_id"])

        mathdial_cadidate_row_ids = mathdial_problem_index.get(sv_record["question"], []) # question -> list of mathdial row ids with the same stepverifyquestion

        if not mathdial_cadidate_row_ids:
            match_records.append(
                {
                    "step_verify_row_id": stepverify_row_id,
                    "question": sv_record["question"],
                    "mathdial_candidate_row_ids_count": 0,
                    "mathdial_row_id": None,
                    "matched_mathdial_question": None,
                    "conversation_score": None,
                    "second_best_score": None,
                    "conversation_margin": None,
                    "score_mathdial_source": None,
                    "student_pair_scores": [],
                    "status": "no_mathdial_question_candidate",
                }
            )
            continue

        scored_candidate_mathdial_rows: list[dict[str, Any]] = []

        for mathdial_row_id in mathdial_cadidate_row_ids:    # among all the mathdial row ids with the same question, get the rows with the same questions
            md_record = mathdial_row_lookup[mathdial_row_id] # mathdial row id -> mathdial record in a dictionary format corresponding to the mathdial_row_id
            mean_pair_score, score_source, pair_scores = score_student_turn_similarity(sv_record, md_record)  # calculate the similarity between the student utterances in Stepverify and Mathdial with the same question
            scored_candidate_mathdial_rows.append(
                {
                    "mathdial_row_id": mathdial_row_id,
                    "score": mean_pair_score,
                    "score_source": score_source,
                    "pair_scores": pair_scores,
                }
            )

        # Deterministic tie-breaking: lower MathDial row ID wins an exact tie.
        scored_candidate_mathdial_rows.sort(key=lambda scored_candidate_mathdial_row: (-scored_candidate_mathdial_row["score"], scored_candidate_mathdial_row["mathdial_row_id"])) # sort cnadidate mathdial rows by score in the descending order and mathdial row id in the ascending order

        best_scored_candidate_mathdial_row = scored_candidate_mathdial_rows[0]
        second_best_scored_candidate_mathdial_row = scored_candidate_mathdial_rows[1]["score"] if len(scored_candidate_mathdial_rows) > 1 else None
        margin_between_best_and_second_best = best_scored_candidate_mathdial_row["score"] - second_best_scored_candidate_mathdial_row if second_best_scored_candidate_mathdial_row is not None else None

        best_selected_mathdial_record = mathdial_row_lookup[best_scored_candidate_mathdial_row["mathdial_row_id"]]

        match_records.append(
            {
                "stepverify_row_id": stepverify_row_id,

                "mathdial_row_id": best_scored_candidate_mathdial_row["mathdial_row_id"],
                "matched_question_in_mathdial": best_selected_mathdial_record["question"],
                "number_of_question_in_mathdial": len(mathdial_cadidate_row_ids),

                "best_student_utterence_similarity_score": best_scored_candidate_mathdial_row["score"],
                "second_best_similarity_score": second_best_scored_candidate_mathdial_row,
                "difference__best_and_second_best": margin_between_best_and_second_best,

                "score_source": best_scored_candidate_mathdial_row["score_source"],

                "student_utterance_pairwise_socres": best_scored_candidate_mathdial_row["pair_scores"],
                "status": "matched"
            }
        )

    student_utterance_similarity_score_result_df = pd.DataFrame.from_records(match_records)
    student_utterance_similarity_score_result_df["mathdial_row_id"] = student_utterance_similarity_score_result_df["mathdial_row_id"].astype(int)
    return student_utterance_similarity_score_result_df

In [ ]:
student_conversation_matches_df = match_student_conversation_between_stepverify_mathdial(sv_df_with_context, mathdial_question_index, mathdial_row_lookup)
student_conversation_matches_df.sort_values(by="matched_question_in_mathdial")

In [ ]:
import numpy as np
import pandas as pd

bins = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.99, 1.000001]
labels = ["0.0~0.2", "0.2~0.4", "0.4~0.6", "0.6~0.8", "0.8~0.9", "0.9~0.99", "0.99~1.0"]

score_bin = pd.cut(
    student_conversation_matches_df[
        "best_student_utterence_similarity_score"
    ],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False,
)
count_by_bin = score_bin.value_counts(dropna=False).sort_index()

print(count_by_bin)
print("total count:", count_by_bin.sum())

## 8. Stage 2 : Match Every StepVerify teacher turn

In [ ]:
# first_match_result = student_conversation_matches_df.iloc[0].to_dict()
# pprint({int(first_match_result["stepverify_row_id"]): first_match_result})

In [ ]:
# sv_df_with_context.head(1)

In [ ]:
# md_df_with_context["teacher_turn_contexts"][0]

In [ ]:
def match_stepverify_teacher_turns(
    sv_df_with_context: pd.DataFrame,
    student_conversation_matches_df: pd.DataFrame,
    mathdial_row_lookup: dict[int, dict[str, Any]],
    threshold: float
) -> pd.DataFrame:
    """ Based on the student conversation matching results, this function first retrieves the matched StepVerify row IDs together with the corresponding MathDial row IDs and related metadata.
        md_conversation_match_lookup: uses each stepverify_row_id as a key and stores the matched MathDial metadata, such as the MathDial row ID, as its value.

        During the loop, the function goes through all rows in the teacher_turn_contexts column of the StepVerify dataset. It uses each StepVerify row ID to retrieve the corresponding MathDial row ID if StepVerify row ID was already is in the previous student conversation match result.
        If a matching MathDial row ID exists, the function retrieves the corresponding MathDial row. If there is no matching MathDial row ID, it saves the base record without any MathDial information.

        From the retrieved MathDial row, the function accesses the each entry in the teacher_turn_contexts column. Each entry in this column contains a teacher utterance, its preceding and following student utterances, and its pedagogy label.
        Only teacher turns with pedagogy labels are used as MathDial teacher context candidates (md_teacher_context_candidates).
        For each teacher-turn context in the current StepVerify teacher_turn_contexts column, the function compares it with all entries with teacher labels in the current MathDial tacher_turn_contexts column  and calculates similarity scores.
        Based on these scores, it selects the MathDial teacher-turn context with the highest mean similarity score. The selected record includes the matched teacher utterance, its pedagogy label, and the similarity scores.

        Finally, the function updates the MathDial-related fields in the base record.
        The match is marked as accepted(True) with the key value of accepted, only when the mean teacher-context similarity score is equal to or greater than the given threshold.

        The function continues until it accesses all the stepverify rows and associated teacher_turn_contexts column of MathDial dataset.
    """

    md_conversation_match_lookup = { int(conversation_match_result["stepverify_row_id"]): conversation_match_result for conversation_match_result in student_conversation_matches_df.to_dict(orient="records") } # stepverify row id : matched mathdial row id and matched record based on the conversation level scores

    turn_match_records: list[dict[str, Any]] = []

    for sv_record in sv_df_with_context.to_dict(orient="records"):
        stepverify_row_id = int(sv_record["stepverify_row_id"])
        md_student_conversation_match_record = md_conversation_match_lookup[stepverify_row_id]
        mathdial_row_id = md_student_conversation_match_record["mathdial_row_id"]

        for sv_teacher_context in sv_record["teacher_turn_contexts"]:
            base_record = {
                "stepverify_row_id": stepverify_row_id,
                "turn_idx__sv_teacher_context": int(sv_teacher_context["turn_idx"]),
                "teacher_text__sv_teacher_context": sv_teacher_context["text"], # teacher utterance
                "prev_student__sv_teacher_context": sv_teacher_context["prev_student"],  # prev student utterance per each teacher utterance in the current conversatoin
                "next_student__sv_teacher_context": sv_teacher_context["next_student"],  # next student utterance per each teacher utterance in the current conversatoin

                # below is to be updated using mathdial teacher_turn contexts
                "mathdial_row_id": None,
                "mathdial_turn_idx__md_teacher_context": None,
                "mathdial_teacher_text__md_teacher_context": None,
                "mathdial_label__md_teacher_context": None,

                "student_conversation_match_score": md_student_conversation_match_record["best_student_utterence_similarity_score"],

                "teacher_text_score__md_sv_teacher_context": None,
                "prev_student_score__md_sv_teacher_context": None,
                "next_student_score__md_sv_teacher_context": None,

                "mean_teacher_context_score": None,
                "threshold": threshold,

                "accepted": False,
                "status": md_student_conversation_match_record["status"]
            }

            # error handling; if the match record does not contain mathdial_row_id, add default dictionary
            if pd.isna(mathdial_row_id):
                turn_match_records.append(base_record)
                continue

            mathdial_row_id = int(mathdial_row_id)
            md_record = mathdial_row_lookup[mathdial_row_id] # fetch mathdial record based on the mathdial row id from matched result

            # A mathdial teacher turn without pedagogy labels cannot transfer a label, so only teacher turns with the labels are considered
            md_teacher_context_candidates = [teacher_turn for teacher_turn in md_record["teacher_turn_contexts"] if teacher_turn.get("label")]

            if not md_teacher_context_candidates:
                base_record["mathdial_row_id"] = mathdial_row_id
                base_record["status"] = "no_labeled_mathdial_teacher_turn"
                turn_match_records.append(base_record)
                continue

            scored_md_teacher_context_candidates: list[dict[str, Any]] = []

            for md_teacher_context in md_teacher_context_candidates:
                score_parts = score_teacher_turn_candidate(sv_teacher_context, md_teacher_context)
                scored_md_teacher_context_candidates.append({
                    "mathdial_teacher_context": md_teacher_context,
                    **score_parts # score between prev student utterances, score between next student utterances, score between teacher utterances, mean score
                    })

            scored_md_teacher_context_candidates.sort(key=lambda row: ( -row["mean_teacher_context_score"], row["mathdial_teacher_context"]["turn_idx"] ))
            best_scored_md_teacher_context = scored_md_teacher_context_candidates[0]
            best_md_teacher_context = best_scored_md_teacher_context["mathdial_teacher_context"]

            accepted: bool = best_scored_md_teacher_context["mean_teacher_context_score"] >= threshold

            base_record.update(
                {
                    "mathdial_row_id": mathdial_row_id,
                    "mathdial_turn_idx__md_teacher_context": best_md_teacher_context["turn_idx"],
                    "mathdial_teacher_text__md_teacher_context":  best_md_teacher_context["text"],
                    "mathdial_label__md_teacher_context": best_md_teacher_context["label"],

                    "teacher_text_score__md_sv_teacher_context": best_scored_md_teacher_context["teacher_text_score"],
                    "prev_student_score__md_sv_teacher_context": best_scored_md_teacher_context["prev_student_score"],
                    "next_student_score__md_sv_teacher_context": best_scored_md_teacher_context["next_student_score"],
                    "mean_teacher_context_score": best_scored_md_teacher_context["mean_teacher_context_score"],

                    "accepted": accepted,
                    "status": "accepted" if accepted else "below_threshold"
                }
            )
            turn_match_records.append(base_record)

    result_df = pd.DataFrame.from_records(turn_match_records)

    for column in ["mathdial_row_id", "mathdial_turn_idx__md_teacher_context"]:
        result_df[column] = result_df[column].astype("Int64")

    return result_df



In [ ]:
teacher_turn_matches_df = match_stepverify_teacher_turns(
    sv_df_with_context,
    student_conversation_matches_df,
    mathdial_row_lookup,
    threshold=0.9
)

teacher_turn_matches_df

In [ ]:
teacher_turn_matches_df["accepted"].value_counts()

In [ ]:
import numpy as np
import pandas as pd

bins = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.99, 1.000001]
labels = ["0.0~0.2", "0.2~0.4", "0.4~0.6", "0.6~0.8", "0.8~0.9", "0.9~0.99", "0.99~1.0"]

score_bin = pd.cut(
    teacher_turn_matches_df["mean_teacher_context_score"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False,
)
count_by_bin = score_bin.value_counts(dropna=False).sort_index()

print(count_by_bin)
print("total count:", count_by_bin.sum())

## Stage 3 : Apply only accepted labels

In [ ]:
from copy import deepcopy

def apply_accepted_pedagogy_labels(
    sv_df_with_context: pd.DataFrame,
    teacher_turn_matches_df: pd.DataFrame,
) -> list[dict[str, Any]]:
    """This function applies the accepted pedagogy labels found by the previous teacher-turn matching function to the original StepVerify dataset.
        It first selects only the teacher-turn matches whose accepted value is True.
        For each accepted match, it uses the stepverify_row_id and teacher turn index to locate the corresponding teacher turn in the StepVerify conversation.
        If the located teacher turn does not already have a pedagogy label, the function assigns the matched MathDial pedagogy label to it.
        Finally, it returns a copied version of the StepVerify DataFrame containing the newly transferred pedagogy labels.
    """
    labeled_data = sv_df_with_context.copy()
    accepted_matches = teacher_turn_matches_df[teacher_turn_matches_df["accepted"]]

    for match in accepted_matches.to_dict(orient="records"):
        stepverify_row_id = int(match["stepverify_row_id"])
        stepverify_teacher_turn_idx = int(match["turn_idx__sv_teacher_context"])
        label = match["mathdial_label__md_teacher_context"]

        target__stepverify_row__conversation_teacher_turn = labeled_data.iloc[stepverify_row_id]["conversation"][stepverify_teacher_turn_idx]

        if target__stepverify_row__conversation_teacher_turn is None:
            continue

        if not target__stepverify_row__conversation_teacher_turn.get("pedagogy"):
            target__stepverify_row__conversation_teacher_turn["pedagogy"] = label

        # pprint(target_turn)

    return labeled_data

labeled_stepverify = apply_accepted_pedagogy_labels(sv_df_with_context, teacher_turn_matches_df)

In [ ]:
labeled_stepverify_ = labeled_stepverify.to_dict(orient="records")
labeled_stepverify_[0]

In [ ]:
len(labeled_stepverify_)

## Summarize the run

In [ ]:
student_conversation_matches_df["best_student_utterence_similarity_score"].isna().any()

In [ ]:
def calculate_bins(series: pd.Series) -> dict[str, int]:
    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.99, 1.000001]
    labels = ["0.0~0.2", "0.2~0.4", "0.4~0.6", "0.6~0.8", "0.8~0.9", "0.9~0.99", "0.99~1.0"]

    score_bin = pd.cut(
        series,
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=False,
    )
    count_by_bin = score_bin.value_counts(dropna=False).sort_index()

    return {
        str(bin_label): int(count)
        for bin_label, count in count_by_bin.items()
    }

def summarize_matching_run(
    student_conversation_matches_df: pd.DataFrame,
    teacher_turn_matches_df: pd.DataFrame
) -> dict[str, Any]:

    accepted_df = teacher_turn_matches_df[teacher_turn_matches_df["accepted"]]
    label_distribution = accepted_df["mathdial_label__md_teacher_context"].value_counts(dropna=False).to_dict()

    student_conversation_similiary_summary = {
        "n_student_conversation_similiary_result": int(len(student_conversation_matches_df)),
        "student_conversation_similarity_result_status": {str(key): int(value) for key, value in student_conversation_matches_df["status"].value_counts(dropna=False).items()},
        "student_conversation_similarity_score_distribution": calculate_bins(student_conversation_matches_df["best_student_utterence_similarity_score"]),
    }

    teacher_turn_match_summary = {
        "threshold_to_accept": 0.9,
        "n_matched_teacher_turns": int(len(teacher_turn_matches_df)),
        "matched_teacher_turns_result_status": {str(k): int(v) for k, v in teacher_turn_matches_df["status"].value_counts(dropna=False).items()},
        "n_accepted_matched_teacher_turns": int(accepted_df.shape[0]),
        "n_rejected_matched_teacher_turns": int(teacher_turn_matches_df.shape[0] - accepted_df.shape[0]),
        "accepted_matched_teacher_turns_distribution": calculate_bins(accepted_df["mean_teacher_context_score"]) if not accepted_df.empty else {},
        "matched_teacher_turns__label_distribution": {str(key): int(value) for key, value in label_distribution.items()}
    }

    return student_conversation_similiary_summary, teacher_turn_match_summary

student_conversation_similiary_summary, teacher_turn_match_summary = summarize_matching_run(
    student_conversation_matches_df,
    teacher_turn_matches_df,
)

# print("student_conversation_similiary_summary:")
# print(json.dumps(student_conversation_similiary_summary, ensure_ascii=False, indent=2))
# print("\n\n teacher_turn_match_summary:")
# print(json.dumps(teacher_turn_match_summary, ensure_ascii=False, indent=2))

## Save Labeled Data and Summaries

In [ ]:
def modify_column_names(labeled_stepverify: pd.DataFrame) -> pd.DataFrame :
    """
        modify columns to align with the original data structure
    """
    columns_to_rename = {"question": "problem", "conversation": "dialog_history",}
    columns_to_drop = ['parsed_conversation_turns', 'teacher_turn_contexts', 'n_parsed_conversation_turns', 'n_student_turns', 'n_teacher_turn_contexts', 'stepverify_row_id', 'student_turns']
    final_labeled_stepverify= labeled_stepverify.rename(columns=columns_to_rename).drop(columns=columns_to_drop, errors="ignore")
    return final_labeled_stepverify

def rename_student_mistake_types(labeled_stepverify_df: pd.DataFrame) -> pd.DataFrame:
    """
        rename student mistake types to align with the original data structure
    """
    # Apply normalize_text and replace spaces with underscores to the 'error_category' column
    labeled_stepverify_df['error_category'] = labeled_stepverify_df['error_category'].apply(normalize_text).apply(lambda x: x.replace(" ", "_"))
    return labeled_stepverify_df

def lowercase_dialogue_texts(labeled_stepverify_df: pd.DataFrame) -> pd.DataFrame:
    """
        lowercase dialogue texts
    """
    labeled_stepverify_df['dialog_history'] = labeled_stepverify_df['dialog_history'].apply(lambda turns: [
        {key: value.lower() if isinstance(value, str) else value for key, value in turn.items() }
        for turn in turns
        ])
    return labeled_stepverify_df

def drop_conversations_with_at_least_one_pedagogy_label(labeled_stepverify_df: pd.DataFrame) -> pd.DataFrame:
    """
        drop rows with conversations where label is not assigned to all teacher turns in the conversation
    """
    incomplete_indices = labeled_stepverify_df.index[ labeled_stepverify_df["dialog_history"].apply(lambda history: any(turn.get("user") == "teacher" and "pedagogy" not in turn for turn in history)) ]
    labeled_stepverify_df = labeled_stepverify_df.drop(incomplete_indices).reset_index(drop=True)
    return labeled_stepverify_df

def process_labeled_stepverify(labeled_stepverify: pd.DataFrame) -> pd.DataFrame:
    final_labeled_stepverify = modify_column_names(labeled_stepverify)
    final_labeled_stepverify = rename_student_mistake_types(final_labeled_stepverify)
    final_labeled_stepverify = lowercase_dialogue_texts(final_labeled_stepverify)
    final_labeled_stepverify = drop_conversations_with_at_least_one_pedagogy_label(final_labeled_stepverify)
    return final_labeled_stepverify

final_labeled_stepverify = process_labeled_stepverify(labeled_stepverify)

print("columns: ")
print(final_labeled_stepverify.columns)
print("\n\nerror categories: ")
print(final_labeled_stepverify['error_category'].unique())
print("\n\nfirst dialogue history:")
pprint(final_labeled_stepverify['dialog_history'][0])

In [ ]:
len(final_labeled_stepverify)

In [ ]:
from pathlib import Path
import math
DATA_OUTPUT_DIR = Path("./data")
META_OUTPUT_DIR = Path("./data/meta_data")

def make_json_safe(value: Any) -> Any:
    """Recursively convert pandas/numpy values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if value is pd.NA:
        return None

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)

    if isinstance(value, float) and math.isnan(value):
        return None

    return value

# def choose_run_suffix(output_dir: Path, threshold: float) -> str:
#     """Return a threshold-based suffix that does not overwrite an existing run."""
#     threshold_tag = f"{threshold:g}"
#     suffix_number = 0

#     while True:
#         suffix = (
#             threshold_tag
#             if suffix_number == 0
#             else f"{threshold_tag}_{suffix_number}"
#         )
#         candidate = output_dir / f"stepverify_labeled_{suffix}.json"

#         if not candidate.exists():
#             return suffix

#         suffix_number += 1

def save_matching_run(
    labeled_stepverify: list[dict[str, Any]],
    student_conversation_matches: list[dict[str, Any]],
    teacher_turn_matches: list[dict[str, Any]],
    student_conversation_similiary_summary: pd.DataFrame,
    teacher_turn_match_summary: pd.DataFrame,
    data_output_dir: Path=DATA_OUTPUT_DIR,
    meta_output_dir: Path=META_OUTPUT_DIR,
    threshold: float=0.9
) -> dict[str, Path]:
    data_output_dir.mkdir(parents=True, exist_ok=True)
    meta_output_dir.mkdir(parents=True, exist_ok=True)

    # data_run_suffix = choose_run_suffix(data_output_dir, threshold)
    # meta_run_suffix = choose_run_suffix(meta_output_dir, threshold)

    labeled_data_path = data_output_dir / f"stepverify_labeled_{threshold}.json"

    student_conversation_match_path = meta_output_dir / f"stepverify_{threshold}__student_converation_match.json"
    teacher_turn_matches_path = meta_output_dir / f"stepverify_{threshold}__teacher_turn_match.json"

    student_conversation_similiary_summary_path = meta_output_dir / f"stepverify_{threshold}__student_conversation_similiary_summary.json"
    teacher_turn_match_summary_path = meta_output_dir / f"stepverify_{threshold}__teacher_turn_match_summary.json"


    with labeled_data_path.open("w", encoding="utf-8") as file:
        json.dump(make_json_safe(labeled_stepverify), file, ensure_ascii=False, indent=2)

    with student_conversation_match_path.open("w", encoding="utf-8") as file:
        json.dump(student_conversation_matches, file, ensure_ascii=False, indent=2)

    with teacher_turn_matches_path.open("w", encoding="utf-8") as file:
        json.dump(teacher_turn_matches, file, ensure_ascii=False, indent=2)

    with student_conversation_similiary_summary_path.open("w", encoding="utf-8") as file:
        json.dump(student_conversation_similiary_summary, file, ensure_ascii=False, indent=2)

    with teacher_turn_match_summary_path.open("w", encoding="utf-8") as file:
        json.dump(teacher_turn_match_summary, file, ensure_ascii=False, indent=2)



    return {
        "labeled_stepverify": labeled_data_path,
        "student_conversation_matches": student_conversation_match_path,
        "teacher_turn_matches": teacher_turn_matches_path,
        "student_conversation_similiary_summary": student_conversation_similiary_summary_path,
        "teacher_turn_match_summary": teacher_turn_match_summary_path,
    }

save_paths = save_matching_run(
    final_labeled_stepverify.to_dict(orient="records"),
    student_conversation_matches_df.to_dict(orient="records"),
    teacher_turn_matches_df.to_dict(orient="records"),
    student_conversation_similiary_summary,
    teacher_turn_match_summary,
)
print(save_paths)

----

In [ ]:
rows_missing_pedagogy = []

for index, row in final_labeled_stepverify.iterrows():
    dialog_history = row['dialog_history']
    first_teacher_turn = None
    for turn in dialog_history:
        # Assuming 'user' is normalized to 'teacher' or 'student' already
        if turn.get('user') == 'teacher':
            first_teacher_turn = turn
            break

    if first_teacher_turn and 'pedagogy' not in first_teacher_turn:
        rows_missing_pedagogy.append({
            'row_index': index,
            'first_teacher_turn': first_teacher_turn
        })

if rows_missing_pedagogy:
    print(f"Number of rows where the first teacher turn is missing 'pedagogy' label: {len(rows_missing_pedagogy)}")
    print("Details:")
    for item in rows_missing_pedagogy:
        print(f"  Row index: {item['row_index']}, First teacher turn: {item['first_teacher_turn']}")
else:
    print("All first teacher turns in every row have a 'pedagogy' label.")

In [ ]:
def filter_complete_pedagogy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter out rows to keep only those where all teacher turns in
    dialog_history have a 'pedagogy' label.
    """
    complete_rows = []
    for index, row in df.iterrows():
        is_complete = True
        for turn in row['dialog_history']:
            if turn.get('user') == 'teacher' and 'pedagogy' not in turn:
                is_complete = False
                break
        if is_complete:
            complete_rows.append(row)

    return pd.DataFrame(complete_rows).reset_index(drop=True)

# Create a complete dataset where all pedagogy labels are assigned
filtered_labeled_stepverify = filter_complete_pedagogy(final_labeled_stepverify)

print(f"Number of data before filtering: {len(final_labeled_stepverify)}")
print(f"Number of data after filtering: {len(filtered_labeled_stepverify)}")
print(f"Number of excluded data: {len(final_labeled_stepverify) - len(filtered_labeled_stepverify)}")

In [ ]:
incomplete_indices = final_labeled_stepverify.index[ final_labeled_stepverify["dialog_history"].apply(lambda history: any(turn.get("user") == "teacher" and "pedagogy" not in turn for turn in history)) ]
print(len(incomplete_indices))
print(incomplete_indices)